# get **twittes** 📬

In [ ]:
from google.colab import files

files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d pokeash/bitcoin-tweets-dataset-20252026
!unzip bitcoin-tweets-dataset-20252026.zip

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/pokeash/bitcoin-tweets-dataset-20252026
License(s): CC0-1.0
100% 670M/670M [00:32<00:00, 21.9MB/s]

Archive:  bitcoin-tweets-dataset-20252026.zip
  inflating: bitcoin_tweets_latest.csv  

In [2]:
import pandas as pd

df = pd.read_csv('bitcoin_tweets_latest.csv',  on_bad_lines='skip')
df.head()

/tmp/ipykernel_3485/549140985.py:3: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('bitcoin_tweets_latest.csv',  on_bad_lines='skip')


,user_name,user_location,user_description,user_created,user_followers,user_friends,user_favourites,user_verified,date,text,hashtags,source,is_retweet
0,DeSota Wilson,"Atlanta, GA","Biz Consultant, real estate, fintech, startups...",2009-04-26 20:05:09,8534.0,7605,4838,False,2026-02-10 23:59:04,Blue Ridge Bank shares halted by NYSE after #b...,['bitcoin'],Twitter Web App,False
1,Tdlmatias,"London, England","IM Academy : The best #forex, #SelfEducation, ...",2014-11-10 10:50:37,128.0,332,924,False,2026-02-10 23:54:48,"Guys evening, I have read this article about B...",NaN,Twitter Web App,False
2,Crypto is the future,NaN,I will post a lot of buying signals for BTC tr...,2019-09-28 16:48:12,625.0,129,14,False,2026-02-10 23:54:33,$BTC A big chance in a billion! Price: \487264...,"['Bitcoin', 'FX', 'BTC', 'crypto']",dlvr.it,False
3,Alex Kirchmaier 🇦🇹🇸🇪 #FactsSuperspreader,Europa,Co-founder @RENJERJerky | Forbes 30Under30 | I...,2016-02-03 13:15:55,1249.0,1472,10482,False,2026-02-10 23:54:06,This network is secured by 9 508 nodes as of t...,['BTC'],Twitter Web App,False
4,ZerrBenz™ ⚔ ✪ 20732,"Bkk, Thailand",I'm a cat slave 🐱 Interested in Blockchain · T...,2010-01-12 07:00:04,742.0,716,2444,False,2026-02-10 23:53:30,💹 Trade #Crypto on #Binance \n\n📌 Enjoy #Cashb...,"['Crypto', 'Binance', 'Cashback']",Twitter Web App,False


In [3]:
df.drop(columns=[
    'user_location','is_retweet',
    'source','user_favourites','user_friends' , 'user_description'] , inplace=True , axis=1)

In [4]:
df.user_followers = pd.to_numeric(df.user_followers, errors='coerce')
df.user_followers.fillna(df['user_followers'].mean(), inplace=True)

/tmp/ipykernel_3485/1501286652.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df.user_followers.fillna(df['user_followers'].mean(), inplace=True)


In [5]:
df.date = pd.to_datetime(df.date, errors='coerce')
df.dropna(subset=['date'], inplace=True)

In [6]:
df = df[df.user_followers > 250_000]

In [7]:
df.dropna(inplace=True)

In [8]:
df.drop('user_created' , inplace=True , axis=1)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18960 entries, 2564 to 5144696
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   user_name       18960 non-null  object        
 1   user_followers  18960 non-null  float64       
 2   user_verified   18960 non-null  object        
 3   date            18960 non-null  datetime64[ns]
 4   text            18960 non-null  object        
 5   hashtags        18960 non-null  object        
dtypes: datetime64[ns](1), float64(1), object(4)
memory usage: 1.0+ MB


In [10]:
df_final = df[['date' , 'text' , 'user_name']]

In [11]:
df_final['date'] = df_final['date'].dt.date
df_final['date'] = pd.to_datetime(df_final['date'])

/tmp/ipykernel_3485/903373489.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['date'] = df_final['date'].dt.date
/tmp/ipykernel_3485/903373489.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['date'] = pd.to_datetime(df_final['date'])


In [12]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18960 entries, 2564 to 5144696
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       18960 non-null  datetime64[ns]
 1   text       18960 non-null  object        
 2   user_name  18960 non-null  object        
dtypes: datetime64[ns](1), object(2)
memory usage: 592.5+ KB


In [13]:
df_final.to_csv('df_final.csv')

# Sentiment Analysis 🔤

In [2]:
from transformers import pipeline
from tqdm import tqdm
import pandas as pd

In [3]:
df_final = pd.read_csv('df_final.csv')
df_final.head()

,Unnamed: 0,date,text,user_name
0,2564,2026-02-10,"Novogratz Predicts, #Bitcoin Price Will Hit $1...",FortuneZ News
1,3635,2026-02-09,"You guys can take #Bitcoin up to $220,000 *rig...",Max Keiser
2,4007,2026-02-09,"#BTC rallies to extend Elon Musk Tesla gains, ...",Daily Express
3,4169,2026-02-09,"#BTC rallies to extend Elon Musk Tesla gains, ...",Daily Express
4,4366,2026-02-09,Seems that #Bitcoin is the magical word right ...,CoinMarketCap


In [4]:
pd.to_datetime(df_final['date']).describe()

,date
count,18960
mean,2025-08-17 01:52:10.632911360
min,2025-03-06 00:00:00
25%,2025-06-21 00:00:00
50%,2025-08-09 00:00:00
75%,2025-10-27 00:00:00
max,2026-03-02 00:00:00


In [5]:

model = "cardiffnlp/twitter-roberta-base-sentiment-latest"

sentiment_task = pipeline("sentiment-analysis", model=model )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
sent_results = {}
for i, d in tqdm(df_final.iterrows(), total=len(df_final)):
    # Ensure the text is a string and not empty
    tweet_text = str(d["text"])
    if not tweet_text.strip(): # Skip empty or whitespace-only strings
        sent_results[d["text"]] = [] # Assign an empty list for skipped entries
        continue
    sent = sentiment_task(tweet_text, truncation=True, max_length=512)
    sent_results[d["text"]] = sent

 16%|█▌        | 3075/18960 [13:43<1:10:51,  3.74it/s]


KeyboardInterrupt: 

In [ ]:
df_final["sentiment"] = df_final["text"].map(
    lambda x: sent_results[x][0]["label"] if sent_results[x] else None
)

df_final["score"] = df_final["text"].map(
    lambda x: sent_results[x][0]["score"] if sent_results[x] else None
)

In [ ]:
df_final.head()

In [ ]:
result = df_final[['user_name' , 'date' , 'text' , 'score' , 'sentiment']]

In [ ]:
result.to_csv('Sentiment_Twitte_For_BTC_250000.csv')

# Get BTC price 🪙